#Stage 1 - Exploratory Data Analysis
#Problem statement
'Saap,’ a smartphone retailer, seeks to analyze historical sales data of previously launched smartphones to identify key trends, patterns, and factors influencing their performance in the market. The objective of this study is to perform detailed statistical analysis on the existing dataset to uncover sales dynamics, seasonal trends, and customer behavior patterns. By leveraging data visualization techniques and effective data storytelling, we aim to translate raw data into meaningful business insights. These findings will enable the retailer to refine its pricing,marketing, and product positioning strategies, ensuring better alignment with market demand and enhancing overall business performance


**Objective:** Analyze historical smartphone data to identify sales dynamics, seasonal trends, and customer behavior patterns. Translate raw data into business insights using statistical tests and visualizations.





In [1]:
#Import libraries
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt


In [2]:
#import dataset
from google.colab import files
uploaded = files.upload()


ModuleNotFoundError: No module named 'google'

In [ ]:
# Load the dataset


df = pd.read_csv('cellphone_data.csv')
print('Rows:', len(df), '| Columns:', df.shape[1])

df.info()

**Perform Other Data Pre-processing steps below such as checking data description, null values etc.**

In [ ]:


#Data Pre-Processing
# 1) Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# 2) Duplicates removal
print("\nDuplicate rows:", df.duplicated().sum())
# df.drop_duplicates(inplace=True)

# 3) Clean text columns
text_cols = ['brand','model','operating system','user_name','Region(City)','gender','occupation']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].str.strip()

# 4) Convert date & create year/month
if 'release date' in df.columns:
    df['release date'] = pd.to_datetime(df['release date'], errors='coerce')
    df['release_year'] = df['release date'].dt.year
    df['release_month'] = df['release date'].dt.month

# 5) Convert numeric columns
num_cols = ['rating','internal memory','RAM','performance','main camera','selfie camera',
            'battery size','screen size','weight','price(INR)','Salary_in_INR','age']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 6) Fill missing values (median for numeric, mode for categorical)
for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)


display(df.head())


**Key Steps & Insights of Data Pre-processing**

1.Used df.info() and df.describe() to confirm column data types and spot anomalies.

2.Handled missing values

3.Identified nulls and imputed:

4.Numeric → median (robust to outliers)
Categorical → mode (most frequent value).
5.Remove duplicates

6.Cleaned text columns

7.Trimmed spaces in brand, model, operating system, Region(City), etc.
Prevents category duplication (e.g., “Android” vs “Android ”).


8.Converted release date to datetime and created release_year and release_month for seasonal analysis.



9.Flagged out-of-range values (e.g., RAM > 64 GB, battery size < 500 mAh).





# Split the variables in the dataset into numerical and categorical types.

In [ ]:

# Split variables into numerical and categorical types
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(exclude=['number']).columns.tolist()


max_len = max(len(numeric_cols), len(categorical_cols))
numeric_cols += [''] * (max_len - len(numeric_cols))
categorical_cols += [''] * (max_len - len(categorical_cols))

summary_df = pd.DataFrame({
    'Numeric Columns': numeric_cols,
    'Categorical Columns': categorical_cols
})

# Display the table
print(summary_df.to_string(index=False))



# Create boxplots for all numerical variables to check for outliers.

In [ ]:

numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
id_like_cols = [c for c in ['user_id', 'cellphone_id'] if c in df.columns]
plot_cols = [c for c in numeric_cols if c not in id_like_cols]

if len(plot_cols) == 0:
    raise ValueError("No numeric columns found to plot after excluding identifier-like columns.")

# -----------------------
# 3) Order columns by IQR (descending) to surface variability first
# -----------------------
def calc_iqr(series: pd.Series) -> float:
    s = series.dropna()
    if s.empty:
        return 0.0
    q1 = np.percentile(s, 25)
    q3 = np.percentile(s, 75)
    return float(q3 - q1)

plot_cols = sorted(plot_cols, key=lambda c: calc_iqr(df[c]), reverse=True)

# -----------------------
# 4) Create grid of boxplots + annotate outliers
# -----------------------
n = len(plot_cols)
cols = 3                                  # number of plots per row
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4), constrained_layout=True)
axes = np.array(axes).reshape(-1)

for i, col in enumerate(plot_cols):
    ax = axes[i]
    sns.boxplot(y=df[col], ax=ax, color='steelblue')
    ax.set_title(f'Boxplot: {col}', fontsize=11)
    ax.set_xlabel("")                     # cleaner look
    ax.grid(True, axis='y', alpha=0.2)

    # Compute outlier counts (1.5 × IQR rule) and annotate
    s = df[col].dropna()
    if len(s) > 0:
        q1 = np.percentile(s, 25)
        q3 = np.percentile(s, 75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        out_low = int((s < lower).sum())
        out_high = int((s > upper).sum())

        # Annotation at bottom-right of each subplot
        ax.text(0.98, 0.02,
                f'Outliers: low={out_low}, high={out_high}',
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=9, color='dimgray')

# Remove extra empty subplots if any
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Global title & caption
fig.suptitle('Boxplots for Numeric Columns (1.5×IQR whiskers; dots beyond = outliers)', fontsize=16, y=1.02)
fig.text(0.5, -0.03,
         'Box = IQR (Q1–Q3), line = median. Taller boxes → higher variability. '
         'Points outside whiskers are outliers by the 1.5×IQR rule.',
         ha='center', va='center', fontsize=10, alpha=0.85)

# Save figure (also shows inline in Colab if you call plt.show())
plt.savefig('boxplots_cellphone_data.png', dpi=150, bbox_inches='tight')
plt.show()

# -----------------------
# 5) Build numeric summary per column (CSV)
# -----------------------
summary_rows = []
for col in plot_cols:
    s = df[col].dropna()
    if len(s) == 0:
        summary_rows.append({
            'column': col, 'count': 0,
            'q1': np.nan, 'median': np.nan, 'q3': np.nan, 'iqr': np.nan,
            'lower_whisker_limit': np.nan, 'upper_whisker_limit': np.nan,
            'outliers_below': 0, 'outliers_above': 0
        })
        continue

    q1 = np.percentile(s, 25)
    q2 = np.percentile(s, 50)
    q3 = np.percentile(s, 75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    summary_rows.append({
        'column': col,
        'count': int(s.size),
        'q1': float(q1),
        'median': float(q2),
        'q3': float(q3),
        'iqr': float(iqr),
        'lower_whisker_limit': float(lower),
        'upper_whisker_limit': float(upper),
        'outliers_below': int((s < lower).sum()),
        'outliers_above': int((s > upper).sum())
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('boxplot_summary_cellphone_data.csv', index=False)

print("Saved: boxplots_cellphone_data.png")
print("Saved: boxplot_summary_cellphone_data.csv")
print("Numeric columns plotted:", plot_cols)




**Intepretition**

**Ratings**
- Mostly high (median ~7), moderate spread (IQR ~4), no extreme outliers.
- Slight left skew → users tend to give higher ratings.

**Internal Memory**
- Dominated by 128 GB; behaves like categorical tiers (64/128/256/512).
- Outliers at both ends → premium and entry-level segments.

**RAM**
- Median ~8 GB, common range 4–8 GB; few 12 GB devices.
- Slight right skew → performance-focused models exist.

**Performance Score**
- Median ~6.8, consistent spread, mild left skew.
- Indicates overall strong performance across devices.

**Main Camera**
- Median ~50 MP, wide range, strong right skew.
- High outliers (64/108 MP) → flagship differentiation.

**Selfie Camera**
- Median ~12 MP, right skew with outliers (16 MP+).
- Creator-focused models push upper tail.

**Battery Size**
- Median ~4600 mAh, cluster near 5000 mAh.
- Few low outliers → older/compact devices.

**Screen Size**
- Median ~6.5″, tight spread; outliers on both ends.
- Compact (<5.4″) and foldable (>7″) are niche.

**Weight**
- Median ~203 g, mild right skew.
- Light outliers (~141 g) vs heavy (~240 g) → design trade-offs.

**Price**
- Median ₹43.7k, very wide IQR, strong right skew.
- Premium devices dominate upper tail (>₹86k).

**Salary**
- Median ₹4.86 L, mild right skew.
- No extreme outliers → typical urban income range.

**Age**
- Median ~33 yrs, right skew.
- More older users than very young → professional segment.
"""




In [ ]:
#1) Correlation of selected specs with price
cols = ['price(INR)','RAM','internal memory','main camera','selfie camera','battery size','performance']
corr_with_price = df[cols].corr()['price(INR)'].sort_values(ascending=False)
print("Correlation with price(INR):")
print(corr_with_price)


**How to interpret:**

Values near +1 show strong positive association (e.g., performance, RAM).
Near 0 means weak/no linear relationship (often battery size).
Negative suggests inverse relation (rare here).

# Compare price by OS (group means/medians)


In [ ]:
df.groupby('operating system')['price(INR)'].agg(['count','mean','median']).sort_values('mean', ascending=False)

**If iOS mean/median > Android: reinforces the pricing premium narrative.**

# Generate a heatmap for all numerical variables to examine their correlations.

In [ ]:

# Choose numerical columns and exclude identifiers
num_cols = df.select_dtypes(include=['number']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['user_id','cellphone_id']]

# Pearson correlation matrix
corr = df[num_cols].corr(method='pearson')

# Plot heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap (Pearson) for Numeric Variables', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap_cellphone_data.png', dpi=150, bbox_inches='tight')
plt.show()

# Save correlation matrix to CSV
corr.to_csv('correlation_matrix_cellphone_data.csv')


# Quick insights

**1.Performance vs. RAM/Main Camera:** Expect positive correlations—higher‑spec phones often score better. (Check the exact coefficients in the CSV.)

**2.Price(INR) tends to correlate positively with features like performance, camera MP, and sometimes weight (larger batteries/screens)**

**3.Battery size vs. weight: Typically positive, as bigger batteries add mass**.

**4.Age/Salary_in_INR**: These are user attributes; their correlations with device specs may be weak/moderate and context‑dependent. Use with caution in modeling.

# Create count plots for all categorical variables to visualize their distributions.

In [ ]:

import math
import re
# 2) Identify categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

# Helper: collapse very high-cardinality columns into Top N + 'Other'
def top_n_counts(series: pd.Series, n=15):
    s = series.fillna('Missing')
    counts = s.value_counts()
    if len(counts) <= n:
        return counts
    top = counts.iloc[:n].copy()
    top['Other'] = counts.iloc[n:].sum()
    return top

# Helper: make a safe filename from the column name
def safe_name(name: str) -> str:
    name = name.replace(' ', '_')
    name = re.sub(r'[^A-Za-z0-9_()-]+', '', name)  # strip odd chars
    return name

# 3) Generate one count plot per categorical column
for col in cat_cols:
    counts = top_n_counts(df[col], n=15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=counts.index, y=counts.values, color='steelblue')
    plt.title(f'Count Plot: {col} (Top 15 + Other if applicable)')
    plt.ylabel('Count')
    plt.xlabel(col)
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, axis='y', alpha=0.2)
    plt.tight_layout()

    fname = f"countplot_{safe_name(col)}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

# 4) a small dashboard grid for key categoricals
key_cols = [c for c in ['brand','operating system','region(city)','gender','occupation'] if c in cat_cols]
if key_cols:
    cols = 3
    rows = math.ceil(len(key_cols)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4), constrained_layout=True)
    axes = np.array(axes).reshape(-1)
    for i, col in enumerate(key_cols):
        counts = top_n_counts(df[col], n=15)
        sns.barplot(x=counts.index, y=counts.values, ax=axes[i], color='steelblue')
        axes[i].set_title(col)
        axes[i].set_ylabel('Count')
        axes[i].set_xlabel('')
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].grid(True, axis='y', alpha=0.2)
    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])
    plt.savefig('countplots_key_categoricals.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:

insights_text = """
# 📌 Strategic Insights

1. **Android-first strategy**
   - ~83% Android → prioritize testing, app features, and support for Android.
   - iOS still matters for premium features and high-value segments.

2. **Model coverage**
   - Ensure device labs/test matrices cover top repeated models:
     Moto G Play (2021), Galaxy A32, 11T Pro, Pixel 6, iPhone 13 Pro Max.

3. **Launch-cycle alignment**
   - Inventory concentrated around 2021–2022 launches.
   - Refresh older SKUs and align training/support content with release waves.

4. **Regional planning**
   - Even allocation across Delhi/Kolkata/Hyderabad.
   - Extra uplift in Chennai and western metros (Mumbai/Pune/Ahmedabad).
   - Targeted campaigns for Bangalore to boost representation.
"""

print(insights_text)


# Create pair plots for all numerical variables to visualize their relationships.

In [ ]:



# 2) Select numeric columns, excluding identifiers
num_cols = df.select_dtypes(include=['number']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['user_id', 'cellphone_id']]
print(f"Numeric columns ({len(num_cols)}): {num_cols}")

# 3) Create pair plots (lower triangle only to keep it readable)
sns.set(style='whitegrid')
pair = sns.pairplot(
    df[num_cols],
    diag_kind='kde',                  # KDE on diagonals
    corner=True,                      # show only lower triangle
    plot_kws={'alpha': 0.6, 's': 20}  # transparency & point size
)
pair.fig.suptitle('Pair Plots for Numeric Variables', y=1.02)

# 4) Save and show
pair.savefig('pairplots_numeric_variables.png', dpi=150)
plt.show()


# 1. Hypothesis Testing
# 1a. Two-sample t-test: Salary difference between genders

In [ ]:

# Filter salaries by gender
male_salary   = df[df['gender'] == 'Male']['Salary_in_INR']
female_salary = df[df['gender'] == 'Female']['Salary_in_INR']

# Independent two-sample t-test (equal variances assumption)
t_stat, p_value = stats.ttest_ind(male_salary, female_salary)

print("T-test: Salary difference between genders")
print("t-statistic:", t_stat, "p-value:", p_value, "\n")


# Interpretation

If p_value < 0.05, there’s statistically significant evidence of a salary difference between Male and Female groups in your sample.
Consider also reporting group means and confidence intervals for clarity

# 1b. One-sample t-test: RAM > 16 GB

In [ ]:



# One-sample, one-sided t-test: H0: mean_RAM = 16; H1: mean_RAM > 16
t_stat, p_value = stats.ttest_1samp(df['RAM'], 16, alternative='greater')

print("One-sample t-test: Mean RAM > 16 GB")
print("t-statistic:", t_stat, "p-value:", p_value, "\n")


**We tested whether the average RAM is greater than 16 GB. The result shows the opposite: the average RAM is significantly less than 16 GB. There is no statistical evidence to support the claim that mean RAM > 16 GB**

# 1c. One-sample t-test: Main camera > 12 MP

In [ ]:

# H0: μ_main_camera = 12 MP
# H1: μ_main_camera  > 12 MP
t_stat, p_value = stats.ttest_1samp(df['main camera'], 12, alternative='greater')

print("One-sample t-test: Main camera > 12 MP")
print("t-statistic:", t_stat, "p-value:", p_value, "\n")


t-statistic = 35.64
Extremely large and positive → your sample mean is much greater than 12 MP compared to the hypothesized value.


p-value ≈ 6.46 × 10⁻¹⁸⁰ (essentially zero)
This is far below any conventional significance level (0.05, 0.01, etc.).
→ Reject H₀ with overwhelming evidence.
→ The mean main camera resolution is significantly greater than 12 MP.

# 1d. Two-sample t-test: Price difference between Android and iOS

In [ ]:
# Filter prices by OS
android_price = df[df['operating system'] == 'Android']['price(INR)']
ios_price     = df[df['operating system'] == 'iOS']['price(INR)']

# Independent two-sample t-test (equal variances assumed)
t_stat, p_value = stats.ttest_ind(android_price, ios_price)

print("T-test: Price difference between Android and iOS")
print("t-statistic:", t_stat, "p-value:", p_value, "\n")


# Interpretation

If p_value < 0.05, you have statistically significant evidence of a price difference between Android and iOS in your sample.
Report group means, standard deviations, and possibly confidence intervals for clear business communication.

# 1e. One-sample t-test: Selfie camera > 8 MP

In [ ]:
# H0: μ_selfie_camera = 8 MP
# H1: μ_selfie_camera  > 8 MP
t_stat, p_value = stats.ttest_1samp(df['selfie camera'], 8, alternative='greater')

print("One-sample t-test: Selfie camera > 8 MP")
print("t-statistic:", t_stat, "p-value:", p_value, "\n")

# Interpretation

Null hypothesis (H₀): Mean selfie camera MP = 8
Alternative (H₁): Mean selfie camera MP > 8
If p_value < 0.05, you can conclude the average selfie camera resolution is significantly greater than 8 MP.

# 1f. One-sample t-test: Battery size > 4000 mAh

In [ ]:

# One-sample, one-sided t-test:
# H0: μ_battery = 4000 mAh
# H1: μ_battery > 4000 mAh
t_stat, p_value = stats.ttest_1samp(df['battery size'], 4000, alternative='greater')

print("One-sample t-test: Battery size > 4000 mAh")
print("t-statistic:", t_stat, "p-value:", p_value, "\n")


# Interpretation

Null hypothesis (H₀): Mean battery size = 4000 mAh
Alternative (H₁): Mean battery size > 4000 mAh (right‑tailed test)
If p_value < 0.05, you can conclude the average battery capacity exceeds 4000 mAh.

# 1g. Two-sample t-test: Salary difference across regions (Delhi vs Mumbai)

In [ ]:


if 'Delhi' in df['Region(City)'].unique() and 'Mumbai' in df['Region(City)'].unique():
    delhi_salary  = df[df['Region(City)'] == 'Delhi']['Salary_in_INR']
    mumbai_salary = df[df['Region(City)'] == 'Mumbai']['Salary_in_INR']

    # Two-sample t-test (equal variances assumed)
    t_stat, p_value = stats.ttest_ind(delhi_salary, mumbai_salary)

    print("T-test: Salary difference between Delhi and Mumbai")
    print("t-statistic:", t_stat, "p-value:", p_value, "\n")
else:
    print("One or both cities (Delhi/Mumbai) not found in Region(City).")


# Interpretation
If p_value < 0.05, there’s statistically significant evidence of a salary difference between Delhi and Mumbai in your sample.
Report group means, standard deviations, and confidence intervals for clearer business communication.

# 2. ANOVA Tests

# 2a. Price difference across brands

In [ ]:

# Build a list of price arrays, one per brand
brands_price = [df[df['brand'] == b]['price(INR)'] for b in df['brand'].unique()]

# One-way ANOVA
f_stat, p_value = stats.f_oneway(*brands_price)

print("ANOVA: Price difference across brands")
print("F-statistic:", f_stat, "p-value:", p_value, "\n")


# Interpretation

H₀ (null): All brands have the same mean price.
H₁ (alternative): At least one brand has a different mean price.
If p_value < 0.05, reject H₀ → brand-level mean price differences are statistically significant.

# 2b. Price difference across occupations

In [ ]:
# Build a list of price arrays, one per occupation
occupations_price = [df[df['occupation'] == o]['price(INR)'] for o in df['occupation'].unique()]

# One-way ANOVA
f_stat, p_value = stats.f_oneway(*occupations_price)

print("ANOVA: Price difference across occupations")
print("F-statistic:", f_stat, "p-value:", p_value, "\n")

# Interpretation

H₀ (null): All occupations have the same mean price.
H₁ (alternative): At least one occupation has a different mean price.
If ANOVA is significant (p < 0.05), report group means, standard deviations, and Tukey HSD results to explain which occupations differ and by how much.

# 2c. Performance rating across age groups (<30, 30-50, >50)

In [ ]:

# Define groups by age bands
group1 = df[df['age'] < 30]['performance']
group2 = df[(df['age'] >= 30) & (df['age'] <= 50)]['performance']
group3 = df[df['age'] > 50]['performance']

# One-way ANOVA: H0 = all age groups have the same mean performance
f_stat, p_value = stats.f_oneway(group1, group2, group3)

print("ANOVA: Performance rating across age groups")
print("F-statistic:", f_stat, "p-value:", p_value, "\n")


# Interpretation

H₀ (null): All three age groups have the same mean performance.
H₁ (alternative): At least one age group’s mean performance differs.
If p_value < 0.05, reject H₀ → there is a statistically significant difference in mean performance across age groups.

# 2d. Battery size across brands

In [ ]:
# Build a list of battery-size arrays, one per brand
brands_battery = [df[df['brand'] == b]['battery size'] for b in df['brand'].unique()]

# One-way ANOVA
f_stat, p_value = stats.f_oneway(*brands_battery)

print("ANOVA: Battery size difference across brands")
print("F-statistic:", f_stat, "p-value:", p_value, "\n")


# Interpretation

H₀ (null): All brands have the same mean battery capacity.
H₁ (alternative): At least one brand has a different mean battery capacity.
If p_value < 0.05, reject H₀ → brand-level differences in mean battery size are statistically significant.

# 2e. Screen size across operating systems

In [ ]:
# Build a list of screen-size arrays, one per OS
os_screen = [df[df['operating system'] == os]['screen size'] for os in df['operating system'].unique()]

# One-way ANOVA
f_stat, p_value = stats.f_oneway(*os_screen)

print("ANOVA: Screen size difference across OS")
print("F-statistic:", f_stat, "p-value:", p_value, "\n")


# Interpretation:

H₀ (ANOVA): All OS groups have the same mean screen size.
H₁: At least one OS has a different mean.
If p_value < 0.05, reject H₀ → there is a statistically significant difference in mean screen size between OS groups.

# 2g. RAM across operating systems

In [ ]:
# Build a list of RAM arrays, one per OS
os_ram = [df[df['operating system'] == os]['RAM'] for os in df['operating system'].unique()]

# One-way ANOVA
f_stat, p_value = stats.f_oneway(*os_ram)

print("ANOVA: RAM difference across OS")
print("F-statistic:", f_stat, "p-value:", p_value, "\n")

# Interpretation:

H₀ (ANOVA): All OS groups have the same mean RAM.
H₁: At least one OS has a different mean.
If p_value < 0.05, reject H₀ → there is a statistically significant difference in mean RAM between OS groups.

# 2h. Main camera across brands

In [ ]:
# Build a list of main-camera arrays, one per brand
brands_camera = [df[df['brand'] == b]['main camera'] for b in df['brand'].unique()]

# One-way ANOVA
f_stat, p_value = stats.f_oneway(*brands_camera)

print("ANOVA: Main camera difference across brands")
print("F-statistic:", f_stat, "p-value:", p_value, "\n")


# Interpretation

H₀ (null): All brands have the same mean main camera MP.
H₁ (alternative): At least one brand has a different mean.
If p_value < 0.05, reject H₀ → there are brand‑level differences in average main camera resolution. Then report Tukey results to pinpoint which brands differ and by how much (include group means & CIs for clarity).

# 3. Chi-Square Tests

# 3a. OS choice vs gender

In [ ]:
# Build contingency table: OS × Gender
os_gender_table = pd.crosstab(df['operating system'], df['gender'])

# Chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(os_gender_table)

print("Chi-square: OS choice vs Gender")
print("Chi2:", chi2, "p-value:", p, "df:", dof, "\n")

# Optional: show expected frequencies (same shape as contingency table)
expected_df = pd.DataFrame(expected, index=os_gender_table.index, columns=os_gender_table.columns)
print("Expected frequencies:\n", expected_df)


# Interpretation

H₀ (null): OS choice and Gender are independent (no association).
H₁ (alternative): OS choice and Gender are associated.
If p-value < 0.05, reject H₀ → there is a statistically significant association between Operating System and Gender in your sample

# 3b. Brand choice vs OS

In [ ]:

brand_os_table = pd.crosstab(df['brand'], df['operating system'])
chi2, p, dof, expected = stats.chi2_contingency(brand_os_table)
print("Chi-square: Brand vs OS")
print("Chi2:", chi2, "p-value:", p, "\n")


**'Interpretation:**

Pearson’s Chi‑square test of independence was conducted to examine the relationship between brand choice and operating system. The association was statistically significant, χ²(9) = 990.00, p < 0.001, with Cramér’s V = 1.00, indicating a perfect association. Apple exclusively uses iOS, while all other brands in the dataset use Android.

# 3c. Brand choice vs occupation

In [ ]:

brand_occ_table = pd.crosstab(df['brand'], df['occupation'])
chi2, p, dof, expected = stats.chi2_contingency(brand_occ_table)
print("Chi-square: Brand vs Occupation")
print("Chi2:", chi2, "p-value:", p, "\n")


**Insights**: Certain occupations, such as managers and IT professionals, show a preference for premium brands like Apple, Samsung, and OnePlus, whereas other roles lean toward mid-range brands.

**Chi-square Test Outcome:**

If p < 0.05, the association between brand choice and occupation is statistically significant, meaning occupation influences brand preference.
If p ≥ 0.05, there is no significant relationship, suggesting brand choice is independent of occupation.